## Concept primer: pandas cleaning toolkit

A DataFrame `df` is a table. `df["col"]` is one column (a Series). You clean a column by **reassigning it**: `df["col"] = <cleaned version>`.

| Need | Code |
|---|---|
| look at it | `df.head()`, `df.info()`, `df.isna().sum()`, `df["col"].value_counts(dropna=False)` |
| remove exact duplicate rows | `df = df.drop_duplicates()` |
| keep rows matching a condition | `df = df[df["col"] > 0]` |
| string ops on a whole column | `.str.strip()`, `.str.lower()`, `.str.title()`, `.str.replace("a", "b", regex=False)` (chain them) |
| text → number | `pd.to_numeric(series)` (add `errors="coerce"` to turn junk into NaN) |
| text → date | `pd.to_datetime(series, errors="coerce")` (junk → `NaT`, the date version of NaN) |
| fill missing | `series.fillna(value)`, e.g. `series.fillna(series.median())` |
| recode values | `series.map({"old": new, ...})` (anything not in the dict becomes NaN) |

**When to use median not mean for filling:** skewed data with outliers (income, transaction amounts). One £950k salary drags the mean up; the median barely moves.

**Tiny example:**
```python
s = pd.Series([" Leeds", "LEEDS", "leeds "])
s.str.strip().str.title()      # -> Leeds, Leeds, Leeds
```


## Problem: clean a messy loan-applications file

`data/loans_messy.csv` has one row per loan application: `application_id, application_date, region, employment, income, loan_amount, defaulted`.

It's messy. **Do the steps in order** (later tests depend on earlier ones). Each step has its own test cell. Run it after the step.

**Target: 25 minutes.** Look things up in the primer table above, that's allowed.


In [ ]:
import pandas as pd
df = pd.read_csv("../data/loans_messy.csv")

# Step 0: inspect. Run these, then write in the cell below what's messy (3-5 bullet points).
print(df.shape)
df.info()
print(df.isna().sum())
df.head(15)


_Step 0 - what's messy in this data:_
- 
- 
- 

**Step 1:** remove exact duplicate rows.

In [ ]:
# your code here


In [ ]:
assert len(df) == 480, len(df)
print('step 1 ok')


**Step 2:** a loan amount of 0 or less is impossible. Keep only rows with `loan_amount > 0`.

In [ ]:
# your code here


In [ ]:
assert len(df) == 474, len(df)
assert (df['loan_amount'] > 0).all()
print('step 2 ok')


**Step 3:** `region` has inconsistent case and stray spaces (`' Leeds'`, `'LEEDS '`). Standardise to `Leeds`, `London`, `Manchester`, `Glasgow`.

In [ ]:
# your code here


In [ ]:
assert sorted(df['region'].unique()) == ['Glasgow', 'Leeds', 'London', 'Manchester'], df['region'].unique()
print('step 3 ok')


**Step 4:** `employment` has variants (`'Self Employed'`, `'self employed'`, `'EMPLOYED'` ...) and missing values.
Standardise to exactly `employed`, `self-employed`, `unemployed`, and fill missing with `unknown`.

In [ ]:
# your code here


In [ ]:
assert sorted(df['employment'].unique()) == ['employed', 'self-employed', 'unemployed', 'unknown'], df['employment'].unique()
print('step 4 ok')


**Step 5:** `income` is text: some rows are `'£34,600'`, some `'34600'`, some missing.
Convert it to a **float** column, then fill missing values with the **median** income.

In [ ]:
# your code here


In [ ]:
assert pd.api.types.is_float_dtype(df['income']), df['income'].dtype
assert df['income'].isna().sum() == 0
assert df['income'].median() == 36500.0, df['income'].median()
print('step 5 ok')


**Step 6:** `defaulted` is a mix of `'1'`, `'0'`, `'Y'`, `'N'`, `'yes'`, `'no'`. Convert to an **integer** column of 0/1.
(Tip: check `df['defaulted'].value_counts()` first.)

In [ ]:
# your code here


In [ ]:
assert pd.api.types.is_integer_dtype(df['defaulted']), df['defaulted'].dtype
assert set(df['defaulted'].unique()) == {0, 1}
assert df['defaulted'].sum() == 125, df['defaulted'].sum()
print('step 6 ok')


**Step 7:** convert `application_date` to datetime. Some rows say `'not provided'`: they should become `NaT`, not crash.

In [ ]:
# your code here


In [ ]:
assert pd.api.types.is_datetime64_any_dtype(df['application_date']), df['application_date'].dtype
assert df['application_date'].isna().sum() == 11
print('step 7 ok - data is clean')


## Explain it (assumptions: graded)

Answer in 1 line each:
1. Why fill income with the **median** and not the mean? (Look at `df['income'].describe()`.)
2. In step 2 you **dropped** bad rows. In step 7 you **kept** rows with missing dates. Why is that a reasonable difference?
3. There are a few incomes of £950,000. Would you remove them? What would you check first?


_your answers here_